# Temporal Prediction — All 4 Models

Trains and evaluates all four spatiotemporal prediction models used in the paper:

| Model | Architecture | Key Design |
|-------|-------------|------------|
| **ConvLSTM** | Single ConvLSTM (32ch) + Conv decoder | Convolutional recurrent spatial-temporal correlations |
| **PredRNN++** | 4-layer ST-LSTM (32,64,64,64) + GHU | Dual-memory spatiotemporal recurrence |
| **MetadataFusion** | CNN encoder-decoder + metadata embeddings | FiLM conditioning on protocol metadata |
| **PhyDNet** | PhyCell + residual ConvLSTM | Physics-guided recurrent cell + residual dynamics |

**Task:** Given 2 consecutive input frames, predict the next frame.  
**Variable gaps:** Δt ∈ {1, 2, 3} intervals to capture short and mid-range dynamics.  
**Metrics:** MSE, SSIM, PSNR  
**Split:** 70 / 15 / 15 (group-wise — no leakage across experimental conditions)

In [ ]:
# !pip install scikit-image tifffile pytorch-msssim --quiet

In [ ]:
# Imports
import os, re, random, math, time
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import tifffile
from PIL import Image
import matplotlib.pyplot as plt
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from skimage.metrics import (
    structural_similarity as sk_ssim,
    peak_signal_noise_ratio as sk_psnr,
    mean_squared_error as sk_mse,
)

try:
    from pytorch_msssim import SSIM as MSSSIM
    HAVE_MSSSIM = True
except ImportError:
    HAVE_MSSSIM = False
    print("pytorch_msssim not found — falling back to L1+MSE loss")

In [ ]:
# Config
class CFG:
    csv_path    = "/kaggle/input/slimia-metadata/slimia_metadata.csv"
    ckpt_dir    = "./checkpoints/temporal/"
    output_dir  = "./results/temporal/"

    img_size    = 128
    seq_len     = 2       # context frames
    min_gap     = 1
    max_gap     = 3
    batch_size  = 8
    num_workers = 4
    num_epochs  = 200
    patience    = 40
    lr          = 1e-4
    seed        = 42
    device      = "cuda" if torch.cuda.is_available() else "cpu"

    # MetadataFusion categorical metadata columns
    cat_cols    = ["microscope", "cell_line", "culture_medium",
                   "formation_method", "magnification"]
    emb_dim     = 32
    cont_dim    = 2   # [seeding_density_num, time_delta]

os.makedirs(CFG.ckpt_dir,   exist_ok=True)
os.makedirs(CFG.output_dir, exist_ok=True)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

set_seed(CFG.seed)
print(f"Device: {CFG.device}")

## Data Pipeline

In [ ]:
# Image loader
def load_grayscale(path, size=CFG.img_size):
    """Load any TIFF as a normalised float32 grayscale array in [0,1]."""
    arr = tifffile.imread(path).astype(np.float32)
    if arr.ndim == 3:
        if arr.shape[0] in [1, 3] and arr.shape[0] < arr.shape[-1]:
            arr = np.transpose(arr, (1, 2, 0))
        arr = arr.mean(axis=-1)
    arr = (arr - arr.min()) / (arr.max() - arr.min() + 1e-8)
    img = Image.fromarray((arr * 255).astype("uint8")).resize((size, size))
    return np.array(img, dtype=np.float32) / 255.0


def extract_num(s):
    """Extract numeric seeding density from strings like '2000cells'."""
    m = re.search(r"([0-9]+(?:[.,][0-9]+)?)", str(s))
    return float(m.group(1).replace(",", ".")) if m else 0.0

In [ ]:
# Build samples from CSV 
df = pd.read_csv(CFG.csv_path)
df["timepoint_hour"] = pd.to_numeric(df["timepoint_hour"], errors="coerce")
if "seeding_density_num" not in df.columns:
    df["seeding_density_num"] = df["seeding_density"].apply(extract_num)

GROUP_KEYS = ["microscope","cell_line","culture_medium","formation_method",
               "seeding_density","magnification","biological_rep","technical_rep"]

# Build group → sorted frames
groups = defaultdict(list)
for _, r in df.iterrows():
    key  = tuple(str(r[k]) for k in GROUP_KEYS)
    path = r.get("full_path") or r.get("filename")
    if pd.isna(r["timepoint_hour"]) or path is None: continue
    groups[key].append({
        "time":               int(r["timepoint_hour"]),
        "path":               path,
        "seeding_density_num": float(r["seeding_density_num"]),
        # store cat codes per frame for MetadataFusion
        "meta_raw":           {c: str(r[c]) for c in CFG.cat_cols},
    })

filtered_groups = {k: sorted(v, key=lambda x: x["time"])
                   for k, v in groups.items()
                   if len(v) >= CFG.seq_len + 1}

# Build (group_key, input_frames, target_frame) triples
samples = []
for k, frames in filtered_groups.items():
    n = len(frames)
    for i in range(n - CFG.seq_len):
        inp = frames[i:i + CFG.seq_len]
        # gap = 1
        samples.append((k, inp, frames[i + CFG.seq_len]))
        # variable gaps
        for gap in range(CFG.min_gap, CFG.max_gap + 1):
            j = i + CFG.seq_len + gap - 1
            if j < n:
                samples.append((k, inp, frames[j]))

print(f"Groups: {len(filtered_groups)} | Samples: {len(samples)}")

# Group-wise split (no leakage)
all_groups = list(filtered_groups.keys())
random.shuffle(all_groups)
s1, s2 = int(0.70 * len(all_groups)), int(0.85 * len(all_groups))
train_g, val_g, test_g = set(all_groups[:s1]), set(all_groups[s1:s2]), set(all_groups[s2:])

train_s = [s for s in samples if s[0] in train_g]
val_s   = [s for s in samples if s[0] in val_g]
test_s  = [s for s in samples if s[0] in test_g]
print(f"Train: {len(train_s)} | Val: {len(val_s)} | Test: {len(test_s)}")

In [ ]:
# Build MetadataFusion category maps (needed before dataset creation) 
all_cat_vals = {c: set() for c in CFG.cat_cols}
for k in filtered_groups:
    for ci, c in enumerate(CFG.cat_cols):
        all_cat_vals[c].add(k[GROUP_KEYS.index(c)] if c in GROUP_KEYS else "")

# Simpler: collect directly from df
cat_maps = {}
for c in CFG.cat_cols:
    uniq = sorted(df[c].astype(str).unique())
    cat_maps[c] = {v: i for i, v in enumerate(uniq)}

cat_cardinalities = {c: len(cat_maps[c]) for c in CFG.cat_cols}
print("Cat cardinalities:", cat_cardinalities)

In [ ]:
# Datasets 
class TemporalDataset(Dataset):
    """
    Base dataset for ConvLSTM, PredRNN++, PhyDNet.
    Returns (ctx [T,1,H,W], tgt [1,H,W], cont [2]) where
    cont = [seeding_density_num, delta_t].
    """
    def __init__(self, samples):
        self.samples = samples

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        k, inp, tgt = self.samples[idx]
        ctx_imgs = [load_grayscale(f["path"]) for f in inp]
        tgt_img  = load_grayscale(tgt["path"])
        ctx = torch.stack([torch.tensor(im).unsqueeze(0) for im in ctx_imgs])  # (T,1,H,W)
        tgt_t = torch.tensor(tgt_img).unsqueeze(0)
        delta  = tgt["time"] - inp[-1]["time"]
        cont   = torch.tensor([inp[-1]["seeding_density_num"], float(delta)],
                               dtype=torch.float32)
        return ctx, tgt_t, cont


class MetadataFusionDataset(Dataset):
    """
    Extended dataset for MetadataFusion — also returns categorical codes
    and continuous metadata for FiLM conditioning.
    Returns (imgs [T,H,W], cat_codes [C], cont [2], tgt [1,H,W]).
    """
    def __init__(self, samples):
        self.samples = samples

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        k, inp, tgt = self.samples[idx]
        imgs = np.stack([load_grayscale(f["path"]) for f in inp])  # (T,H,W)
        tgt_img = load_grayscale(tgt["path"])

        cat_codes = np.array(
            [cat_maps[c].get(inp[0]["meta_raw"].get(c, ""), 0)
             for c in CFG.cat_cols], dtype=np.int64)

        delta = tgt["time"] - inp[-1]["time"]
        cont  = np.array([inp[-1]["seeding_density_num"], float(delta)],
                          dtype=np.float32)

        return (torch.tensor(imgs),
                torch.tensor(cat_codes),
                torch.tensor(cont),
                torch.tensor(tgt_img).unsqueeze(0))


def make_loaders(ds_cls, s_train, s_val, s_test):
    kw = dict(batch_size=CFG.batch_size, num_workers=CFG.num_workers, pin_memory=True)
    return (DataLoader(ds_cls(s_train), shuffle=True,  **kw),
            DataLoader(ds_cls(s_val),   shuffle=False, **kw),
            DataLoader(ds_cls(s_test),  shuffle=False, **kw))

train_loader, val_loader, test_loader = make_loaders(
    TemporalDataset, train_s, val_s, test_s)
mf_train, mf_val, mf_test = make_loaders(
    MetadataFusionDataset, train_s, val_s, test_s)

## Model Architectures

In [ ]:
# ConvLSTM 
class ConvLSTMCell(nn.Module):
    def __init__(self, input_dim, hidden_dim, kernel_size=3):
        super().__init__()
        p = kernel_size // 2
        self.conv       = nn.Conv2d(input_dim + hidden_dim, 4 * hidden_dim,
                                    kernel_size, padding=p)
        self.hidden_dim = hidden_dim

    def forward(self, x, h, c):
        i, f, o, g = torch.chunk(self.conv(torch.cat([x, h], 1)), 4, 1)
        c = torch.sigmoid(f) * c + torch.sigmoid(i) * torch.tanh(g)
        h = torch.sigmoid(o) * torch.tanh(c)
        return h, c

    def init_hidden(self, B, H, W, device):
        z = torch.zeros(B, self.hidden_dim, H, W, device=device)
        return z, z.clone()


class ConvLSTM(nn.Module):
    """Single ConvLSTM cell (hidden=32) + 1×1 Conv decoder."""
    def __init__(self, input_dim=1, hidden_dim=32):
        super().__init__()
        self.cell    = ConvLSTMCell(input_dim, hidden_dim)
        self.decoder = nn.Conv2d(hidden_dim, 1, 1)

    def forward(self, x_seq):
        B, T, C, H, W = x_seq.shape
        h, c = self.cell.init_hidden(B, H, W, x_seq.device)
        for t in range(T):
            h, c = self.cell(x_seq[:, t], h, c)
        return self.decoder(h)

In [ ]:
# PredRNN++ 
class SpatioTemporalLSTMCell(nn.Module):
    """
    Spatiotemporal LSTM as in Wang et al. (2018) PredRNN++.
    Maintains two memory states: H (zigzag temporal) and M (spatial).
    """
    def __init__(self, input_dim, hidden_dim, kernel_size=3):
        super().__init__()
        p  = kernel_size // 2
        id = input_dim + hidden_dim
        self.hidden_dim = hidden_dim
        # gates for H memory
        self.conv_h = nn.Conv2d(id, 4 * hidden_dim, kernel_size, padding=p)
        # gates for M memory
        self.conv_m = nn.Conv2d(id, 3 * hidden_dim, kernel_size, padding=p)
        # output gate
        self.conv_o = nn.Conv2d(hidden_dim * 3, hidden_dim, kernel_size, padding=p)

    def forward(self, x, h, c, m):
        xh  = torch.cat([x, h], 1)
        i, g, f, o_h = torch.chunk(self.conv_h(xh), 4, 1)
        i = torch.sigmoid(i); f = torch.sigmoid(f); g = torch.tanh(g)
        c_new = f * c + i * g

        xm = torch.cat([x, m], 1)
        i2, f2, g2 = torch.chunk(self.conv_m(xm), 3, 1)
        i2 = torch.sigmoid(i2); f2 = torch.sigmoid(f2); g2 = torch.tanh(g2)
        m_new = f2 * m + i2 * g2

        o = torch.sigmoid(self.conv_o(torch.cat([c_new, m_new, o_h], 1)))
        h_new = o * torch.tanh(
            nn.functional.conv2d(
                torch.cat([c_new, m_new], 1),
                weight=torch.eye(self.hidden_dim, 2 * self.hidden_dim,
                                  device=x.device).view(self.hidden_dim, 2 * self.hidden_dim, 1, 1),
                padding=0
            )
        )
        return h_new, c_new, m_new

    def init_hidden(self, B, H, W, device):
        z = torch.zeros(B, self.hidden_dim, H, W, device=device)
        return z, z.clone(), z.clone()


class GradientHighwayUnit(nn.Module):
    """GHU bridging the first and second ST-LSTM layers."""
    def __init__(self, hidden_dim, kernel_size=3):
        super().__init__()
        p = kernel_size // 2
        self.conv = nn.Conv2d(hidden_dim * 2, hidden_dim * 2, kernel_size, padding=p)

    def forward(self, x, z):
        p, u = torch.chunk(self.conv(torch.cat([x, z], 1)), 2, 1)
        u = torch.sigmoid(u)
        return u * torch.tanh(p) + (1 - u) * z


class PredRNNPP(nn.Module):
    """
    4-layer Spatiotemporal LSTM with Gradient Highway Unit.
    Channels: [32, 64, 64, 64] as in Table 3 of the paper.
    """
    def __init__(self, input_dim=1, dims=(32, 64, 64, 64)):
        super().__init__()
        self.dims  = dims
        self.cells = nn.ModuleList()
        in_d = input_dim
        for d in dims:
            self.cells.append(SpatioTemporalLSTMCell(in_d, d))
            in_d = d
        self.ghu     = GradientHighwayUnit(dims[0])
        self.decoder = nn.Conv2d(dims[-1], 1, 1)

    def forward(self, x_seq):
        B, T, C, H, W = x_seq.shape
        dev = x_seq.device
        # initialise hidden states
        states = [c.init_hidden(B, H, W, dev) for c in self.cells]
        z = torch.zeros(B, self.dims[0], H, W, device=dev)

        for t in range(T):
            x = x_seq[:, t]
            # layer 0 + GHU
            h0, c0, m0 = self.cells[0](x, *states[0])
            z = self.ghu(h0, z)
            states[0] = (h0, c0, m0)
            # layers 1–3: pass M downward
            inp = z
            for li in range(1, len(self.cells)):
                h, c, m = self.cells[li](inp, states[li][0], states[li][1], states[li][2])
                states[li] = (h, c, m)
                inp = h

        return self.decoder(states[-1][0])

In [ ]:
# MetadataFusion (MMFusionNet) 
def _cb(i, o):
    return nn.Sequential(
        nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(inplace=True),
        nn.Conv2d(o, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(inplace=True))


class MetadataFusion(nn.Module):
    """
    CNN encoder-decoder + FiLM conditioning from protocol metadata.
    Categorical metadata (microscope, cell_line, etc.) are embedded and
    combined with continuous features (seeding_density, Δt) to generate
    per-channel scale (γ) and shift (β) at the bottleneck.
    """
    def __init__(self, seq_len=CFG.seq_len, base_ch=32):
        super().__init__()
        B = base_ch
        # Encoder — input channels = seq_len
        self.e1 = _cb(seq_len, B);     self.p1 = nn.MaxPool2d(2)
        self.e2 = _cb(B,   B*2);       self.p2 = nn.MaxPool2d(2)
        self.e3 = _cb(B*2, B*4);       self.p3 = nn.MaxPool2d(2)
        self.e4 = _cb(B*4, B*8);       self.p4 = nn.MaxPool2d(2)
        bot_ch  = B * 8

        # Metadata embeddings + FiLM
        self.embs = nn.ModuleDict({
            c: nn.Embedding(cat_cardinalities[c], CFG.emb_dim)
            for c in CFG.cat_cols
        })
        meta_in = len(CFG.cat_cols) * CFG.emb_dim + CFG.cont_dim
        self.meta_mlp  = nn.Sequential(nn.Linear(meta_in, 256), nn.ReLU(),
                                        nn.Linear(256, 256), nn.ReLU())
        self.film_gen  = nn.Linear(256, 2 * bot_ch)

        # Decoder with skip connections
        self.u4 = nn.ConvTranspose2d(bot_ch, B*4, 2, 2); self.d4 = _cb(B*4 + B*8, B*4)
        self.u3 = nn.ConvTranspose2d(B*4,   B*2, 2, 2); self.d3 = _cb(B*2 + B*4, B*2)
        self.u2 = nn.ConvTranspose2d(B*2,   B,   2, 2); self.d2 = _cb(B   + B*2, B)
        self.u1 = nn.ConvTranspose2d(B,     B,   2, 2); self.d1 = _cb(B   + B,   B)
        self.final = nn.Sequential(nn.Conv2d(B, 1, 3, padding=1), nn.Sigmoid())

        self._bot_ch = bot_ch

    def forward(self, x_seq, cat_codes, cont):
        # x_seq: (B, T, H, W)  cat_codes: (B, C)  cont: (B, 2)
        x  = x_seq
        e1 = self.e1(x);  p1 = self.p1(e1)
        e2 = self.e2(p1); p2 = self.p2(e2)
        e3 = self.e3(p2); p3 = self.p3(e3)
        e4 = self.e4(p3); b  = self.p4(e4)   # bottleneck

        # FiLM
        embs = torch.cat([self.embs[c](cat_codes[:, i])
                          for i, c in enumerate(CFG.cat_cols)], dim=1)
        m    = self.meta_mlp(torch.cat([embs, cont], dim=1))
        gam, bet = torch.chunk(self.film_gen(m), 2, dim=1)
        b    = b * (1 + gam.view(-1, self._bot_ch, 1, 1)) \
               + bet.view(-1, self._bot_ch, 1, 1)

        d = self.d4(torch.cat([self.u4(b), e4], 1))
        d = self.d3(torch.cat([self.u3(d), e3], 1))
        d = self.d2(torch.cat([self.u2(d), e2], 1))
        d = self.d1(torch.cat([self.u1(d), e1], 1))
        return self.final(d)

In [ ]:
# PhyDNet 
class PhyCell(nn.Module):
    """Learns a bank of spatial operators and integrates du/dt = F(h) via Euler."""
    def __init__(self, channels, num_ops=3, kernel_size=3):
        super().__init__()
        p = kernel_size // 2
        self.ops    = nn.ModuleList([
            nn.Conv2d(channels, channels, kernel_size, padding=p,
                      groups=channels, bias=False)
            for _ in range(num_ops)])
        self.coeffs = nn.Parameter(torch.randn(num_ops, channels, 1, 1) * 0.01)
        self.gate   = nn.Conv2d(channels * 2, channels, 1)

    def forward(self, x, h_prev, dt=1.0):
        ops_out = sum(self.coeffs[k] * op(h_prev) for k, op in enumerate(self.ops))
        du      = torch.tanh(ops_out)
        if isinstance(dt, torch.Tensor):
            dt = dt.view(-1, 1, 1, 1)
        h_tilde = h_prev + dt * du
        g       = torch.sigmoid(self.gate(torch.cat([x, h_tilde], 1)))
        return g * h_tilde + (1 - g) * h_prev


class PhyDNet(nn.Module):
    """
    PhyCell (physics) + residual ConvLSTM cell.
    Encoder projects input to phy_ch + res_ch channels.
    Decoder reconstructs the next frame.
    """
    def __init__(self, enc_ch=64, phy_ch=32, res_ch=32):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(1, enc_ch // 2, 3, padding=1), nn.BatchNorm2d(enc_ch // 2), nn.ReLU(inplace=True),
            nn.Conv2d(enc_ch // 2, enc_ch, 3, padding=1), nn.BatchNorm2d(enc_ch), nn.ReLU(inplace=True),
            nn.Conv2d(enc_ch, phy_ch + res_ch, 1))
        self.phy_ch  = phy_ch
        self.res_ch  = res_ch
        self.phycell = PhyCell(phy_ch)
        self.rescell = ConvLSTMCell(res_ch, res_ch)
        self.decoder = nn.Sequential(
            nn.Conv2d(phy_ch + res_ch, enc_ch, 3, padding=1), nn.BatchNorm2d(enc_ch), nn.ReLU(inplace=True),
            nn.Conv2d(enc_ch, enc_ch // 2, 3, padding=1), nn.BatchNorm2d(enc_ch // 2), nn.ReLU(inplace=True),
            nn.Conv2d(enc_ch // 2, 1, 1), nn.Sigmoid())

    def forward(self, x_seq, dt=None):
        B, T, C, H, W = x_seq.shape
        dev   = x_seq.device
        phy_h = torch.zeros(B, self.phy_ch, H, W, device=dev)
        res_h, res_c = self.rescell.init_hidden(B, H, W, dev)

        for t in range(T):
            z     = self.encoder(x_seq[:, t])
            phy_h = self.phycell(z[:, :self.phy_ch], phy_h,
                                  dt=dt if t == T - 1 else 1.0)
            res_h, res_c = self.rescell(z[:, self.phy_ch:], res_h, res_c)

        return self.decoder(torch.cat([phy_h, res_h], 1))

## Training & Evaluation

In [ ]:
# Metrics
def batch_metrics(pred, tgt):
    p = pred.detach().cpu().numpy()
    t = tgt.detach().cpu().numpy()
    mse_v, ssim_v, psnr_v = [], [], []
    for i in range(p.shape[0]):
        pi = np.clip(p[i, 0], 0, 1)
        ti = t[i, 0]
        mse_v.append(float(sk_mse(ti, pi)))
        try:    ssim_v.append(float(sk_ssim(ti, pi, data_range=1.0)))
        except: ssim_v.append(0.0)
        try:    psnr_v.append(float(sk_psnr(ti, pi, data_range=1.0)))
        except: psnr_v.append(20.0)
    return np.mean(mse_v), np.mean(ssim_v), np.mean(psnr_v)


# Loss 
def make_loss():
    if HAVE_MSSSIM:
        ssim_mod = MSSSIM(data_range=1.0, channel=1).to(CFG.device)
        def loss_fn(p, t):
            return 0.8 * nn.MSELoss()(p, t) + 0.2 * (1 - ssim_mod(p, t))
    else:
        def loss_fn(p, t):
            return 0.6 * nn.MSELoss()(p, t) + 0.4 * nn.L1Loss()(p, t)
    return loss_fn


# Generic training loop 
def train_temporal(model_name, model, t_loader, v_loader,
                   is_metadata_fusion=False):
    """
    Trains a temporal model with early stopping on val SSIM.
    Returns (trained model, history dict).
    """
    model     = model.to(CFG.device)
    loss_fn   = make_loss()
    optimizer = optim.Adam(model.parameters(), lr=CFG.lr)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", patience=10, factor=0.5)
    scaler    = torch.amp.GradScaler(enabled=(CFG.device == "cuda"))

    best_ssim = -1.0; bad = 0
    ckpt      = os.path.join(CFG.ckpt_dir, f"{model_name}_best.pth")
    history   = dict(train_loss=[], val_mse=[], val_ssim=[], val_psnr=[])

    for epoch in range(1, CFG.num_epochs + 1):
        # ── Train ──
        model.train()
        t_loss = 0.0
        for batch in tqdm(t_loader, desc=f"[{model_name}] ep{epoch:03d}", leave=False):
            if is_metadata_fusion:
                imgs, cats, cont, tgt = [b.to(CFG.device) for b in batch]
            else:
                ctx, tgt, cont = [b.to(CFG.device) for b in batch]
                imgs = ctx

            optimizer.zero_grad()
            with torch.amp.autocast(device_type=CFG.device):
                if is_metadata_fusion:
                    pred = model(imgs, cats, cont)
                elif model_name == "PhyDNet":
                    pred = model(imgs, dt=cont[:, 1])
                else:
                    pred = model(imgs)
                loss = loss_fn(pred, tgt)
            scaler.scale(loss).backward()
            scaler.step(optimizer); scaler.update()
            t_loss += loss.item()
        history["train_loss"].append(t_loss / len(t_loader))

        # ── Validate ──
        model.eval()
        v_mse, v_ssim, v_psnr = [], [], []
        with torch.no_grad():
            for batch in v_loader:
                if is_metadata_fusion:
                    imgs, cats, cont, tgt = [b.to(CFG.device) for b in batch]
                    pred = model(imgs, cats, cont)
                else:
                    ctx, tgt, cont = [b.to(CFG.device) for b in batch]
                    imgs = ctx
                    pred = (model(imgs, dt=cont[:, 1]) if model_name == "PhyDNet"
                            else model(imgs))
                m, s, p = batch_metrics(pred, tgt)
                v_mse.append(m); v_ssim.append(s); v_psnr.append(p)

        vm, vs, vp = np.mean(v_mse), np.mean(v_ssim), np.mean(v_psnr)
        history["val_mse"].append(vm)
        history["val_ssim"].append(vs)
        history["val_psnr"].append(vp)

        scheduler.step(vs)
        print(f"  Epoch {epoch:03d} | Loss {history['train_loss'][-1]:.5f} "
              f"| MSE {vm:.4f} | SSIM {vs:.4f} | PSNR {vp:.2f}")

        if vs > best_ssim:
            best_ssim = vs; bad = 0
            torch.save(model.state_dict(), ckpt)
            print(f"  ✓ Saved best (SSIM {best_ssim:.4f})")
        else:
            bad += 1
            if bad >= CFG.patience:
                print("  Early stopping."); break

    return model, history


@torch.no_grad()
def evaluate_temporal(model, loader, model_name, is_metadata_fusion=False):
    model.eval()
    all_mse, all_ssim, all_psnr = [], [], []
    for batch in tqdm(loader, desc=f"Test {model_name}"):
        if is_metadata_fusion:
            imgs, cats, cont, tgt = [b.to(CFG.device) for b in batch]
            pred = model(imgs, cats, cont)
        else:
            ctx, tgt, cont = [b.to(CFG.device) for b in batch]
            pred = (model(ctx, dt=cont[:, 1]) if model_name == "PhyDNet"
                    else model(ctx))
        m, s, p = batch_metrics(pred, tgt)
        all_mse.append(m); all_ssim.append(s); all_psnr.append(p)
    return {"mse": np.mean(all_mse), "ssim": np.mean(all_ssim), "psnr": np.mean(all_psnr)}

## Run All Models

In [ ]:
# Train & evaluate
MODEL_REGISTRY = [
    ("ConvLSTM",       ConvLSTM(),                False, train_loader, val_loader, test_loader),
    ("PredRNN++",      PredRNNPP(),               False, train_loader, val_loader, test_loader),
    ("MetadataFusion", MetadataFusion(),           True,  mf_train,     mf_val,     mf_test),
    ("PhyDNet",        PhyDNet(),                  False, train_loader, val_loader, test_loader),
]

all_results = {}

for name, model, is_mf, tl, vl, tel in MODEL_REGISTRY:
    print(f"\n{'='*60}\n  {name}\n{'='*60}")
    set_seed(CFG.seed)
    model, history = train_temporal(name, model, tl, vl, is_mf)

    # Reload best checkpoint
    ckpt = os.path.join(CFG.ckpt_dir, f"{name}_best.pth")
    model.load_state_dict(torch.load(ckpt, map_location=CFG.device))
    test_m = evaluate_temporal(model, tel, name, is_mf)
    all_results[name] = test_m
    print(f"  Test → MSE {test_m['mse']:.4f} | SSIM {test_m['ssim']:.4f} "
          f"| PSNR {test_m['psnr']:.2f}")

In [ ]:
# Results Table (Table 7 in paper)
rows = [{"Model": k, **{m: f"{v:.4f}" for m, v in r.items()}}
        for k, r in all_results.items()]
res_df = pd.DataFrame(rows).set_index("Model")
print("\nTemporal Prediction Results")
print(res_df.to_string())
res_df.to_csv(os.path.join(CFG.output_dir, "temporal_results.csv"))

## Visualisation & Temporal Saliency

In [ ]:
# Qualitative comparison: all 4 models on the same batch
def visualize_all_models(batch, models_dict, n=4):
    ctx, tgt, cont = batch
    ctx  = ctx[:n].to(CFG.device)
    tgt  = tgt[:n].to(CFG.device)
    cont = cont[:n].to(CFG.device)

    n_cols = CFG.seq_len + 1 + len(models_dict)
    fig, axes = plt.subplots(n, n_cols, figsize=(n_cols * 2, n * 2))

    for i in range(n):
        col = 0
        for t in range(CFG.seq_len):
            axes[i, col].imshow(ctx[i, t, 0].cpu(), cmap="gray", vmin=0, vmax=1)
            if i == 0: axes[i, col].set_title(f"Input t-{CFG.seq_len - t}", fontsize=8)
            axes[i, col].axis("off"); col += 1

        axes[i, col].imshow(tgt[i, 0].cpu(), cmap="gray", vmin=0, vmax=1)
        if i == 0: axes[i, col].set_title("Ground Truth", fontsize=8)
        axes[i, col].axis("off"); col += 1

        for mname, mmodel in models_dict.items():
            with torch.no_grad():
                if mname == "PhyDNet":
                    pred = mmodel(ctx, dt=cont[:, 1])
                else:
                    pred = mmodel(ctx)
            axes[i, col].imshow(pred[i, 0].cpu().clamp(0, 1), cmap="gray")
            if i == 0: axes[i, col].set_title(mname, fontsize=8)
            axes[i, col].axis("off"); col += 1

    plt.suptitle("Temporal Prediction — All Models", fontsize=11)
    plt.tight_layout()
    plt.savefig(os.path.join(CFG.output_dir, "temporal_predictions.png"))
    plt.show()


def temporal_saliency(model, x_seq, tgt, model_name):
    """Perturbation-based frame importance: drop in SSIM when each frame is zeroed."""
    with torch.no_grad():
        base = model(x_seq).squeeze().cpu().numpy()
    tgt_np = tgt.squeeze().cpu().numpy()
    base_s = sk_ssim(tgt_np, np.clip(base, 0, 1), data_range=1.0)
    scores = []
    for t in range(x_seq.shape[1]):
        x_m = x_seq.clone(); x_m[:, t] = 0.0
        with torch.no_grad():
            pred_m = model(x_m).squeeze().cpu().numpy()
        s_m = sk_ssim(tgt_np, np.clip(pred_m, 0, 1), data_range=1.0)
        scores.append(max(base_s - s_m, 0))
    return scores

In [ ]:
# Load best checkpoints for all non-fusion models
vis_models = {}
for name, factory_cls in [("ConvLSTM", ConvLSTM),
                           ("PredRNN++", PredRNNPP),
                           ("PhyDNet",  PhyDNet)]:
    m = factory_cls().to(CFG.device)
    m.load_state_dict(torch.load(
        os.path.join(CFG.ckpt_dir, f"{name}_best.pth"),
        map_location=CFG.device))
    m.eval()
    vis_models[name] = m

# Get one test batch
sample_batch = next(iter(test_loader))
visualize_all_models(sample_batch, vis_models)

# Frame importance plot (Fig 6 in paper)
ctx_s, tgt_s, _ = sample_batch
ctx_s = ctx_s[:1].to(CFG.device)
tgt_s = tgt_s[:1]

fig, axes = plt.subplots(1, 3, figsize=(10, 3))
for ax, (name, m) in zip(axes, vis_models.items()):
    scores = temporal_saliency(m, ctx_s, tgt_s, name)
    ax.bar(range(len(scores)), scores, color="steelblue")
    ax.set_title(f"{name} — Frame Importance", fontsize=9)
    ax.set_xlabel("Frame index")
    ax.set_ylabel("SSIM drop")
plt.tight_layout()
plt.savefig(os.path.join(CFG.output_dir, "frame_importance.png"))
plt.show()